In [2]:
_datamode = "llm4poi/ca/dataset_gowalla_ca_ne"


In [3]:
import os
import ast
import re
import pandas as pd
from datetime import timedelta
from openlocationcode import openlocationcode as olc

# Filter out users and POIs with low frequency
def do_filter(df, poi_min_freq=10, user_min_freq=10):
    df = df.copy()

    df['PoiFreq'] = df.groupby('Pid')['Uid'].transform('count')
    df = df[df['PoiFreq'] >= poi_min_freq]

    df['UserFreq'] = df.groupby('Uid')['Pid'].transform('count')
    df = df[df['UserFreq'] >= user_min_freq]

    df = df.drop(columns=['PoiFreq', 'UserFreq'])
    return df


def get_pluscode(latitude, longitude):
    plus_code = olc.encode(latitude, longitude)
    return plus_code[:6]


def parse_catname(value):
    if pd.isna(value):
        return 'Unknown'
    text = str(value)

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list) and parsed and isinstance(parsed[0], dict):
            return str(parsed[0].get('name', 'Unknown'))
    except Exception:
        pass

    match = re.search(r"'name'\s*:\s*'([^']+)'", text)
    if match:
        return match.group(1)

    return text


dataset = _datamode.strip().replace('\\', '/')
file_name = f"datasets/{dataset}.csv"
dataset_name = os.path.basename(dataset)
out_dir = f"datasets/{dataset}"
os.makedirs(out_dir, exist_ok=True)

raw_df = pd.read_csv(file_name)

In [6]:
raw_df.head()

,UserId,PoiId,PoiCategoryId,Latitude,Longitude,UTCTime
0,0,19542,"[{'url': '/categories/45', 'name': 'Airport'}]",37.616356,-122.38615,2010-08-19T21:59:09Z
1,0,19542,"[{'url': '/categories/45', 'name': 'Airport'}]",37.616356,-122.38615,2010-06-24T14:27:35Z
2,0,19542,"[{'url': '/categories/45', 'name': 'Airport'}]",37.616356,-122.38615,2010-06-06T18:48:32Z
3,4,19542,"[{'url': '/categories/45', 'name': 'Airport'}]",37.616356,-122.38615,2010-06-19T15:37:36Z
4,8,19542,"[{'url': '/categories/45', 'name': 'Airport'}]",37.616356,-122.38615,2010-10-07T03:20:57Z


In [7]:
import ast

# 先解析 PoiCategoryId
raw_df["PoiCategoryParsed"] = raw_df["PoiCategoryId"].apply(ast.literal_eval)

# 取第一个元素中的 url 和 name
raw_df["category_url"] = raw_df["PoiCategoryParsed"].apply(
    lambda x: x[0].get("url") if isinstance(x, list) and len(x) > 0 else None
)

raw_df["category_name"] = raw_df["PoiCategoryParsed"].apply(
    lambda x: x[0].get("name") if isinstance(x, list) and len(x) > 0 else None
)

# 看结果
print(raw_df[["PoiCategoryId", "category_url", "category_name"]].head())

                                    PoiCategoryId    category_url  \
0  [{'url': '/categories/45', 'name': 'Airport'}]  /categories/45   
1  [{'url': '/categories/45', 'name': 'Airport'}]  /categories/45   
2  [{'url': '/categories/45', 'name': 'Airport'}]  /categories/45   
3  [{'url': '/categories/45', 'name': 'Airport'}]  /categories/45   
4  [{'url': '/categories/45', 'name': 'Airport'}]  /categories/45   

  category_name  
0       Airport  
1       Airport  
2       Airport  
3       Airport  
4       Airport  


In [8]:
raw_df.head()

,UserId,PoiId,PoiCategoryId,Latitude,Longitude,UTCTime,PoiCategoryParsed,category_url,category_name
0,0,19542,"[{'url': '/categories/45', 'name': 'Airport'}]",37.616356,-122.38615,2010-08-19T21:59:09Z,"[{'url': '/categories/45', 'name': 'Airport'}]",/categories/45,Airport
1,0,19542,"[{'url': '/categories/45', 'name': 'Airport'}]",37.616356,-122.38615,2010-06-24T14:27:35Z,"[{'url': '/categories/45', 'name': 'Airport'}]",/categories/45,Airport
2,0,19542,"[{'url': '/categories/45', 'name': 'Airport'}]",37.616356,-122.38615,2010-06-06T18:48:32Z,"[{'url': '/categories/45', 'name': 'Airport'}]",/categories/45,Airport
3,4,19542,"[{'url': '/categories/45', 'name': 'Airport'}]",37.616356,-122.38615,2010-06-19T15:37:36Z,"[{'url': '/categories/45', 'name': 'Airport'}]",/categories/45,Airport
4,8,19542,"[{'url': '/categories/45', 'name': 'Airport'}]",37.616356,-122.38615,2010-10-07T03:20:57Z,"[{'url': '/categories/45', 'name': 'Airport'}]",/categories/45,Airport


In [ ]:
# url 和 name 的分布一致说明是同一个东西，即 Category

print("url 分布：")
print(raw_df["category_url"].value_counts(dropna=False))

print("\nname 分布：")
print(raw_df["category_name"].value_counts(dropna=False))

url 分布：
category_url
/categories/121    28391
/categories/18     23196
/categories/15     21719
/categories/16     19486
/categories/433    19053
                   ...  
/categories/198        3
/categories/421        2
/categories/401        2
/categories/398        2
/categories/12         1
Name: count, Length: 336, dtype: int64

name 分布：
category_name
Corporate Office     28391
Mexican              21719
Asian                19979
American             19486
Starbucks            19053
                     ...  
Independent Event        3
Protest                  2
LIVESTRONG Day           2
Republican Event         2
Entertainment            1
Name: count, Length: 337, dtype: int64


In [10]:


# Gowalla schema: UserId, PoiId, PoiCategoryId, Latitude, Longitude, UTCTime
# 检查当前数据表的列，判断它是否是新版Gowalla（6列主键字段：UserId, PoiId, PoiCategoryId, Latitude, Longitude, UTCTime）
# if {'UserId', 'PoiId', 'PoiCategoryId', 'Latitude', 'Longitude', 'UTCTime'}.issubset(raw_df.columns):
df = raw_df.copy()

df['Region'] = df.apply(lambda row: get_pluscode(row['Latitude'], row['Longitude']), axis=1)

# Keep UTC timestamp but normalize format
dt = pd.to_datetime(df['UTCTime'], utc=True, errors='coerce')
df['Time'] = dt.dt.tz_localize(None).dt.strftime('%Y-%m-%d %H:%M')

df = df.rename(columns={
    'UserId': 'Uid',
    'PoiId': 'Pid',
    'category_name': 'Catname'
})

df = df[['Uid', 'Pid', 'Catname', 'Region', 'Time']]

# Basic cleanup
df = df.dropna(subset=['Uid', 'Pid', 'Catname', 'Region', 'Time'])
df['Uid'] = pd.to_numeric(df['Uid'], errors='coerce').astype('Int64')
df['Pid'] = pd.to_numeric(df['Pid'], errors='coerce').astype('Int64')
df = df.dropna(subset=['Uid', 'Pid'])
df['Uid'] = df['Uid'].astype(int)
df['Pid'] = df['Pid'].astype(int)
df = df.sort_values(by=['Uid', 'Time']).reset_index(drop=True)

filtered_df = do_filter(df, poi_min_freq=10, user_min_freq=10)
filtered_df.to_csv(f"{out_dir}/{dataset_name}.csv", index=False)

print(f"raw shape: {raw_df.shape}")
print(f"filtered shape: {filtered_df.shape}")
print(f"saved to: {out_dir}/{dataset_name}.csv")


raw shape: (636512, 9)
filtered shape: (351857, 5)
saved to: datasets/llm4poi/ca/dataset_gowalla_ca_ne/dataset_gowalla_ca_ne.csv


In [11]:
df.head()

,Uid,Pid,Catname,Region,Time
0,0,19542,Airport,849VJJ,2010-06-06 18:48
1,0,14608,Coffee Shop,849VQH,2010-06-06 22:11
2,0,1221889,Conference,849VQH,2010-06-06 22:40
3,0,86754,Pub,849VQH,2010-06-07 06:01
4,0,1221889,Conference,849VQH,2010-06-07 11:06


## 已经保存好新的 ca 数据，开始处理用新数据来处理

In [13]:
import os
import pandas as pd
import random


dataset = _datamode.strip().replace('\\', '/')
dataset_name = os.path.basename(dataset)
dataset_dir = f"datasets/{dataset}"

random.seed(42)

df = pd.read_csv(f"{dataset_dir}/{dataset_name}.csv")

uids = list(df['Uid'].unique())
pids = list(df['Pid'].unique())
cats = list(df['Catname'].unique())
regs = list(df['Region'].unique())

random.shuffle(uids)
random.shuffle(pids)
random.shuffle(cats)
random.shuffle(regs)

uid_map = {uid: i for i, uid in enumerate(uids, start=1)}
pid_map = {pid: i for i, pid in enumerate(pids, start=1)}
cat_map = {cat: i for i, cat in enumerate(cats, start=1)}
reg_map = {reg: i for i, reg in enumerate(regs, start=1)}

df['Uid'] = df['Uid'].map(uid_map)
df['Pid'] = df['Pid'].map(pid_map)
df['Catname'] = df['Catname'].map(cat_map)
df['Region'] = df['Region'].map(reg_map)

os.makedirs(dataset_dir, exist_ok=True)

pd.DataFrame(list(uid_map.items()), columns=['Original_Uid', 'Mapped_Uid']).to_csv(f"{dataset_dir}/uid_mapping.csv", index=False)
pd.DataFrame(list(pid_map.items()), columns=['Original_Pid', 'Mapped_Pid']).to_csv(f"{dataset_dir}/pid_mapping.csv", index=False)
pd.DataFrame(list(cat_map.items()), columns=['Original_Catname', 'Mapped_Catname']).to_csv(f"{dataset_dir}/catname_mapping.csv", index=False)
pd.DataFrame(list(reg_map.items()), columns=['Original_Region', 'Mapped_Region']).to_csv(f"{dataset_dir}/region_mapping.csv", index=False)

df.to_csv(f"{dataset_dir}/data.csv", index=False)
print(f"saved to: {dataset_dir}/data.csv")


saved to: datasets/llm4poi/ca/dataset_gowalla_ca_ne/data.csv


In [14]:
import os
import pandas as pd
from collections import Counter, defaultdict


dataset = _datamode.strip().replace('\\', '/')
dataset_dir = f"datasets/{dataset}"
file_name = f"{dataset_dir}/data.csv"

df = pd.read_csv(file_name)
df['Time'] = pd.to_datetime(df['Time']).dt.hour

poi_sequence = df.groupby('Uid').agg({
    'Pid': list,
    'Catname': list
}).reset_index()


def get_forward_neighbors(df, column, min_freq=1):
    neighbor_counts = defaultdict(Counter)
    all_pois = set()

    for sequence in df[column]:
        all_pois.update(sequence)
        for i in range(len(sequence) - 1):
            current_poi = sequence[i]
            next_poi = sequence[i + 1]
            neighbor_counts[current_poi][next_poi] += 1

    df_data = []
    for poi in all_pois:
        counter = neighbor_counts.get(poi, {})
        filtered_neighbors = {
            neighbor: freq for neighbor, freq in counter.items() if freq >= min_freq
        }
        if filtered_neighbors:
            sorted_neighbors = [
                neighbor for neighbor, _ in sorted(filtered_neighbors.items(), key=lambda x: x[1], reverse=True)
            ]
        else:
            sorted_neighbors = []
        df_data.append((poi, sorted_neighbors))

    neighbors_df = pd.DataFrame(df_data, columns=[column, 'neighbors'])
    return neighbors_df


def get_neighbors(df, column, min_freq=1):
    neighbor_counts = defaultdict(Counter)
    all_pois = set()

    for sequence in df[column]:
        all_pois.update(sequence)
        for i, poi in enumerate(sequence):
            if i > 0:
                neighbor_counts[poi][sequence[i - 1]] += 1
            if i < len(sequence) - 1:
                neighbor_counts[poi][sequence[i + 1]] += 1

    df_data = []
    for poi in all_pois:
        counter = neighbor_counts.get(poi, {})
        filtered_neighbors = {
            neighbor: freq for neighbor, freq in counter.items() if freq >= min_freq
        }

        sorted_neighbors = [
            neighbor for neighbor, _ in sorted(filtered_neighbors.items(), key=lambda x: x[1], reverse=True)
        ]
        df_data.append((poi, sorted_neighbors))

    neighbors_df = pd.DataFrame(df_data, columns=[column, 'neighbors'])
    return neighbors_df


poi_info = df.groupby('Pid').agg({
    'Uid': list,
    'Catname': lambda x: x.iloc[0],
    'Region': lambda x: x.iloc[0],
    'Time': list
}).reset_index()

poi_info['Uid'] = poi_info['Uid'].apply(lambda uids: [uid for uid, count in Counter(uids).items() if count >= 1])
poi_info['Time'] = poi_info['Time'].apply(lambda times: [time for time, count in Counter(times).items() if count >= 1])

poi_neighbors = get_neighbors(poi_sequence, 'Pid', 1)
poi_info['neighbors'] = poi_info['Pid'].map(poi_neighbors.set_index('Pid')['neighbors'])

forward_neighbors = get_forward_neighbors(poi_sequence, 'Pid', 1)
poi_info['forward_neighbors'] = poi_info['Pid'].map(forward_neighbors.set_index('Pid')['neighbors'])

poi_info.to_csv(f"{dataset_dir}/poi_info.csv", index=False)
print(f"saved to: {dataset_dir}/poi_info.csv")


saved to: datasets/llm4poi/ca/dataset_gowalla_ca_ne/poi_info.csv


In [15]:
info_df = pd.read_csv(f"datasets/llm4poi/ca/dataset_gowalla_ca_ne/poi_info.csv")
info_df.head()

,Pid,Uid,Catname,Region,Time,neighbors,forward_neighbors
0,1,"[6075, 3848, 609, 3090, 5432, 4630, 4249]",54,125,"[6, 4, 3, 19, 15, 22, 1]","[1, 3525, 3276, 11953, 12473, 6751, 1508, 6519...","[1, 11953, 6751, 6519, 229, 8963, 13218, 3472,..."
1,2,"[5370, 5127, 1087, 5529, 4383, 2929, 1139]",201,951,"[19, 1, 5, 3, 0, 4, 2, 21, 17]","[11291, 2145, 13293, 4686, 8542, 11239, 1732, ...","[4686, 11239, 2145, 12028, 911, 5363, 2444, 13..."
2,3,"[3446, 6343, 2133, 3184, 712, 1221, 2871, 4991...",62,680,"[22, 19, 17, 21, 20, 18]","[11606, 3, 4469, 576, 8582, 10505, 11165, 1013...","[11606, 3, 10505, 10139, 9698, 1011, 8495, 446..."
3,4,"[6428, 2134, 5700, 833, 5694, 2030, 4260, 5085...",201,277,"[21, 0, 2, 19, 20, 22, 23, 16]","[1230, 4, 4186, 2736, 12052, 12346, 10295, 202...","[4, 12346, 10295, 1230, 4186, 7968, 7604, 81, ..."
4,5,"[1883, 1265, 4835, 2112, 2510, 4532, 2311, 291...",160,14,"[3, 1, 0, 21, 20, 5, 2, 22, 23]","[5995, 8457, 3813, 7749, 13780, 319, 10170, 86...","[13780, 319, 10170, 7749, 753, 5761, 11893, 38..."


In [16]:
import os
import pandas as pd


dataset = _datamode.strip().replace('\\', '/')
dataset_dir = f"datasets/{dataset}"
file_name = f"{dataset_dir}/data.csv"
df = pd.read_csv(file_name)

df = df[['Uid', 'Pid', 'Time']]
df['Time'] = pd.to_datetime(df['Time'])
df = df.sort_values(by='Time').reset_index(drop=True)

train_size = int(0.8 * len(df))
train_df = df[:train_size]
test_df = df[train_size:]


def romove_users_pois_test(df_train, df_test):
    users_train = df_train['Uid'].unique()
    pois_train = df_train['Pid'].unique()
    df_test = df_test[df_test['Uid'].isin(users_train)]
    df_test = df_test[df_test['Pid'].isin(pois_train)]
    return df_test


test_df = romove_users_pois_test(train_df, test_df)
test_uids = test_df['Uid'].unique()
expanded_df = df[df['Uid'].isin(test_uids)]

train_df.to_csv(f'{dataset_dir}/train_data.csv', index=False)
expanded_df.to_csv(f'{dataset_dir}/test_data.csv', index=False)
print(f"saved to: {dataset_dir}/train_data.csv")
print(f"saved to: {dataset_dir}/test_data.csv")


saved to: datasets/llm4poi/ca/dataset_gowalla_ca_ne/train_data.csv
saved to: datasets/llm4poi/ca/dataset_gowalla_ca_ne/test_data.csv


In [17]:
import os
import pandas as pd
import random


def generate_train_sequences(df: pd.DataFrame, window_size: int, step_size: int, mask_prob: float) -> pd.DataFrame:

    df = df.copy()
    df['Time'] = pd.to_datetime(df['Time'])

    results = []

    for uid, group in df.groupby('Uid'):
        group = group.sort_values('Time').reset_index(drop=True)

        if len(group) > 80:
            group = group.iloc[-80:]

        n = len(group)

        if n < window_size:
            if n >= 10:
                input_pids = group['Pid'].iloc[:-1].tolist()
                input_times = group['Time'].iloc[:-1].tolist()
                target_pid = group['Pid'].iloc[-1]
                target_time = group['Time'].iloc[-1]

                results.append({
                    'Uid': uid,
                    'Pids': input_pids,
                    'Times': input_times,
                    'Target': target_pid,
                    'Target_time': target_time
                })
            continue

        for start in range(n - 1, window_size - 2, -step_size):
            window = group.iloc[start - window_size + 1: start + 1]

            input_pids = window['Pid'].iloc[:-1].tolist()
            input_times = window['Time'].iloc[:-1].tolist()
            original_target_pid = window['Pid'].iloc[-1]
            original_target_time = window['Time'].iloc[-1]

            if random.random() < mask_prob and len(input_pids) >= 1:
                drop_idx = random.randint(0, len(input_pids) - 1)
                target_pid = input_pids[drop_idx]
                target_time = input_times[drop_idx]
                input_pids = input_pids[:drop_idx] + input_pids[drop_idx + 1:] + [original_target_pid]
                input_times = input_times[:drop_idx] + input_times[drop_idx + 1:] + [original_target_time]
            else:
                target_pid = original_target_pid
                target_time = original_target_time

            results.append({
                'Uid': uid,
                'Pids': input_pids,
                'Times': input_times,
                'Target': target_pid,
                'Target_time': target_time
            })

    train = pd.DataFrame(results)

    train['Times'] = train['Times'].apply(lambda x: [t.strftime('%Y-%m-%d %H:%M') for t in x])
    train['Target_time'] = train['Target_time'].dt.strftime('%Y-%m-%d %H:%M')

    return train


def generate_test_sequences(test_df: pd.DataFrame, window_size: int):

    test_df = test_df.copy()
    test_df['Time'] = pd.to_datetime(test_df['Time'])

    val_records = []
    test_records = []

    for uid, group in test_df.groupby('Uid'):
        group = group.sort_values('Time').reset_index(drop=True)
        n = len(group)

        if n < window_size:
            if n > 2:
                test_records.append({
                    'Uid': uid,
                    'Pids': group['Pid'].iloc[:-1].tolist(),
                    'Times': group['Time'].iloc[:-1].tolist(),
                    'Target': group['Pid'].iloc[-1],
                    'Target_time': group['Time'].iloc[-1]
                })
                val_records.append({
                    'Uid': uid,
                    'Pids': group['Pid'].iloc[:-2].tolist(),
                    'Times': group['Time'].iloc[:-2].tolist(),
                    'Target': group['Pid'].iloc[-2],
                    'Target_time': group['Time'].iloc[-2]
                })
            continue

        if n >= window_size + 1:
            val_start = n - window_size - 1
            val_window = group.iloc[val_start:val_start + window_size]
            val_records.append({
                'Uid': uid,
                'Pids': val_window['Pid'].iloc[:-1].tolist(),
                'Times': val_window['Time'].iloc[:-1].tolist(),
                'Target': val_window['Pid'].iloc[-1],
                'Target_time': val_window['Time'].iloc[-1]
            })

        test_window = group.iloc[n - window_size:]
        test_records.append({
            'Uid': uid,
            'Pids': test_window['Pid'].iloc[:-1].tolist(),
            'Times': test_window['Time'].iloc[:-1].tolist(),
            'Target': test_window['Pid'].iloc[-1],
            'Target_time': test_window['Time'].iloc[-1]
        })

    val_df = pd.DataFrame(val_records)
    test_df = pd.DataFrame(test_records)

    for _df in [val_df, test_df]:
        _df['Times'] = _df['Times'].apply(lambda x: [t.strftime('%Y-%m-%d %H:%M') for t in x])
        _df['Target_time'] = _df['Target_time'].dt.strftime('%Y-%m-%d %H:%M')

    return val_df, test_df


dataset = _datamode.strip().replace('\\', '/')
dataset_dir = f"datasets/{dataset}"
os.makedirs(f"{dataset_dir}/data", exist_ok=True)

train = pd.read_csv(f"{dataset_dir}/train_data.csv")
test = pd.read_csv(f"{dataset_dir}/test_data.csv")
all_data = pd.read_csv(f"{dataset_dir}/data.csv")

random.seed(42)
train = generate_train_sequences(train, 50, 10, 0.1)
val, test = generate_test_sequences(test, 50)

val_all, test_all = generate_test_sequences(all_data, 50)
test_all.to_csv(f"{dataset_dir}/data/test_all.csv", index=False)

train.to_csv(f"{dataset_dir}/data/train.csv", index=False)
val.to_csv(f"{dataset_dir}/data/val.csv", index=False)
test.to_csv(f"{dataset_dir}/data/test.csv", index=False)
print(f"saved to: {dataset_dir}/data/*.csv")


saved to: datasets/llm4poi/ca/dataset_gowalla_ca_ne/data/*.csv


In [ ]:
import pandas as pd


dataset = _datamode.strip().replace('\\', '/')
dataset_dir = f"datasets/{dataset}"
file_name = f"{dataset_dir}/data.csv"
df = pd.read_csv(file_name)

df = df[['Uid', 'Pid', 'Time']]

user_history_length = df.groupby('Uid').size()
average_history_length = user_history_length.mean()
print(average_history_length)

history_length_counts = user_history_length.value_counts()
most_frequent_length = history_length_counts.idxmax()
print(most_frequent_length)


53.37636529126213
10


: 